In [ ]:
!git clone  https://github.com/humaidifikri/Credit-Risk-Modelling.git


fatal: destination path 'Credit-Risk-Modelling' already exists and is not an empty directory.


In [ ]:
# Jalankan ini di sel paling atas notebook Anda
!pip install cuml-cu12 --extra-index-url=https://nvidia.com
# Cell pertama — jalankan ini sebelum yang lain
!pip install -q --upgrade imbalanced-learn scikit-learn xgboost optuna

# Restart runtime setelah install
import os
os.kill(os.getpid(), 9)
%load_ext cuml.accel

Looking in indexes: https://pypi.org/simple, https://nvidia.com


In [1]:
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score, RandomizedSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OrdinalEncoder, LabelEncoder, StandardScaler, MinMaxScaler
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, accuracy_score, classification_report, confusion_matrix,precision_recall_curve
)

from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")

In [2]:
# df = pd.read_csv('../data/cleaned_loan_data')
df = pd.read_csv('/content/Credit-Risk-Modelling/data/cleaned_loan_data')

df['term'] = df['term'].astype('str')
df['term'] = df['term'].str.extract(r'(\d+)').astype(int)


df['home_ownership'] = df['home_ownership'].replace(
    {'NONE': 'OTHER', 'ANY': 'OTHER'}
)

nominal_features   = ['home_ownership', 'verification_status', 'initial_list_status']
ordinal_features   = ['grade']
ordinal_categories = [['A', 'B', 'C', 'D', 'E', 'F', 'G']]

num_features = [c for c in df.select_dtypes(include=['int64', 'float64']).columns
                if c != 'target'
                and c not in ordinal_features
                and c not in nominal_features]

target = 'target'

X = df.drop(columns='target')
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
# print(X_train.shape,y_train.shape,X_test.shape,y_test.shape)

RANDOM_STATE = 42
N_ITER= 10
CV_FOLDS= 5

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
cv= StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

In [3]:
numeric_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Ordinal
ordinal_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown='use_encoded_value',
        unknown_value=-1
    ))
])

# Nominal
nominal_pipeline = Pipeline([
#     ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value',
                               unknown_value=-1))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline,  num_features),
    ('ord', ordinal_pipeline,  ordinal_features),
    ('nom', nominal_pipeline,  nominal_features),
], remainder='drop')

In [4]:
# 1. Sesuaikan parameter distribusi (Hapus l1_ratio karena solver qn fokus pada l1/l2)
lr_param_dist = {
    'model__C':        [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0],
    # 'model__penalty':  ['l1', 'l2'],
    'model__max_iter': [500, 1000, 1500, 2000]
}

# 2. Definisikan Pipeline
lr_base = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(sampling_strategy=0.5, random_state=RANDOM_STATE, k_neighbors=5)),
    ('model', LogisticRegression(               # DIUBAH: 'qn' adalah solver bawaan GPU cuML yang sangat cepat
        class_weight='balanced',
        random_state=RANDOM_STATE
        # Parameter n_jobs dihapus karena GPU menangani paralelisme secara internal
    ))
])

# 3. Definisikan Pencarian Parameter
lr_search = RandomizedSearchCV(
    estimator=lr_base,
    param_distributions=lr_param_dist,
    n_iter=N_ITER,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    n_jobs=1,                     # DIUBAH: Set ke 1 saat memakai GPU agar tidak terjadi tabrakan memori (VRAM)
    refit=True
)

# 4. Jalankan Proses Training di GPU
lr_search.fit(X_train, y_train)

print(f"  Best AUC-ROC (CV) : {lr_search.best_score_:.4f}")
print(f"  Best Params:")
for k, v in lr_search.best_params_.items():
    print(f"    {k:<25} : {v}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits


KeyboardInterrupt: 

In [8]:
import optuna

RANDOM_STATE = 42
N_TRIALS     = 50
CV_FOLDS     = 5
SCORING      = 'roc_auc'

def objective_lr(trial):
    params = {
        'C':            trial.suggest_float('C', 1e-4, 100.0, log=True),
        # 'penalty':      trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet']),
        # 'solver':       'saga',   # satu-satunya solver yang support semua penalty
        'max_iter':     trial.suggest_int('max_iter', 500, 2000),
        'class_weight': 'balanced',
        'random_state': RANDOM_STATE,
        # 'n_jobs':       -1,
    }

    # elasticnet butuh l1_ratio
    # if params['penalty'] == 'elasticnet':
    #     params['l1_ratio'] = trial.suggest_float('l1_ratio', 0.0, 1.0)

    model = ImbPipeline([
        ('preprocessor', preprocessor),
        ('smote', SMOTE(sampling_strategy=0.5, random_state=RANDOM_STATE, k_neighbors=5)),
        ('model', LogisticRegression(**params))
    ])

    scores = cross_val_score(
        model, X_train, y_train,
        cv=cv, scoring=SCORING, n_jobs=1
    )
    return scores.mean()


# start = time.time()
study_lr = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_lr.optimize(objective_lr, n_trials=N_TRIALS, show_progress_bar=True)
for k, v in study_lr.best_params.items():
    print(f"    {k:<20} : {v}")


[I 2026-05-22 07:24:30,334] A new study created in memory with name: no-name-8067885f-55d8-4547-8f44-ad2c2ef0f08b


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-05-22 07:25:19,961] Trial 0 finished with value: 0.6766872598588292 and parameters: {'C': 0.017670169402947963, 'max_iter': 1927}. Best is trial 0 with value: 0.6766872598588292.
[I 2026-05-22 07:26:10,358] Trial 1 finished with value: 0.6767350723568025 and parameters: {'C': 2.465832945854912, 'max_iter': 1398}. Best is trial 1 with value: 0.6767350723568025.
[I 2026-05-22 07:26:57,863] Trial 2 finished with value: 0.6759346416416149 and parameters: {'C': 0.0008632008168602544, 'max_iter': 734}. Best is trial 1 with value: 0.6767350723568025.
[I 2026-05-22 07:27:42,183] Trial 3 finished with value: 0.6751202255559479 and parameters: {'C': 0.00022310108018679258, 'max_iter': 1800}. Best is trial 1 with value: 0.6767350723568025.
[I 2026-05-22 07:28:33,407] Trial 4 finished with value: 0.676738199878553 and parameters: {'C': 0.4042872735027334, 'max_iter': 1562}. Best is trial 4 with value: 0.676738199878553.
[I 2026-05-22 07:29:16,436] Trial 5 finished with value: 0.67463952508